# 🟡 Solution: Batch Capsule-Capsule Overlap

**Primitive:** parametric segment-to-segment minimum distance with `np.where` clamping

**Reduction:** `out[i,j]` is True iff `min_{s,t∈[0,1]} ||P1+s·d1 − P3−t·d2|| ≤ r_a[i]+r_b[j]` — solved by gradient zeroing → clamped s → clamped t → recomputed s, all broadcast over N×M.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import numpy as np

In [ ]:
# primitive: parametric segment-to-segment distance

def capsule_overlap(caps_a, caps_b):
    EPS = 1e-12
    P1 = caps_a[:, None, :2]   # (N, 1, 2)
    P2 = caps_a[:, None, 2:4]  # (N, 1, 2)
    P3 = caps_b[None, :, :2]   # (1, M, 2)
    P4 = caps_b[None, :, 2:4]  # (1, M, 2)
    r_sum = caps_a[:, None, 4] + caps_b[None, :, 4]  # (N, M)

    d1 = P2 - P1   # (N, 1, 2)
    d2 = P4 - P3   # (1, M, 2)
    rv = P1 - P3   # (N, M, 2)

    a = (d1 * d1).sum(-1)  # (N, 1)
    e = (d2 * d2).sum(-1)  # (1, M)
    b = (d1 * d2).sum(-1)  # (N, M) — d1 (N,1,2) × d2 (1,M,2)
    c = (d1 * rv).sum(-1)  # (N, M)
    f = (d2 * rv).sum(-1)  # (N, M)

    denom = a * e - b * b  # (N, M)

    # Unconstrained optimal s; parallel segments → s=0
    s = np.where(
        denom > EPS,
        np.clip((b * f - c * e) / np.where(denom > EPS, denom, 1.0), 0.0, 1.0),
        0.0
    )

    # t from s, then clamp
    t_unc = (b * s + f) / np.where(e > EPS, e, 1.0)
    t = np.clip(t_unc, 0.0, 1.0)

    # Recompute s from clamped t, then clamp again
    s = np.clip((b * t - c) / np.where(a > EPS, a, 1.0), 0.0, 1.0)

    cp1 = P1 + s[..., None] * d1   # (N, M, 2)
    cp2 = P3 + t[..., None] * d2   # (N, M, 2)
    dist = np.sqrt(((cp1 - cp2) ** 2).sum(-1))  # (N, M)

    return dist <= r_sum

In [ ]:
# 🔍 Verify solution
caps_a = np.array([[0.0, 0.0, 2.0, 0.0, 1.0]])
caps_b = np.array([[0.0, 1.5, 2.0, 1.5, 1.0],
                   [0.0, 5.0, 2.0, 5.0, 0.5]])
result = capsule_overlap(caps_a, caps_b)
print("shape:", result.shape)   # expect (1, 2)
print("result:", result)         # expect [[True, False]]

In [ ]:
# ✅ Inline test suite
import numpy as np, time

# ── Test 1: touching capsules (dist == r_sum → True) ──────────────────────
a = np.array([[0.0, 0.0, 2.0, 0.0, 1.5]])
b = np.array([[0.0, 3.0, 2.0, 3.0, 1.5]])
r = capsule_overlap(a, b)
assert r.shape == (1, 1), f"Shape: {r.shape}"
assert r[0, 0] == True, "Touching capsules should overlap"
print("Test 1 passed: touching capsules")

# ── Test 2: separated capsules ────────────────────────────────────────────
a2 = np.array([[0.0, 0.0, 2.0, 0.0, 0.5]])
b2 = np.array([[0.0, 5.0, 2.0, 5.0, 0.5]])
r2 = capsule_overlap(a2, b2)
assert r2[0, 0] == False, "Separated capsules should not overlap"
print("Test 2 passed: separated capsules")

# ── Test 3: perpendicular crossing spines ─────────────────────────────────
a3 = np.array([[0.0, 1.0, 3.0, 1.0, 0.1]])
b3 = np.array([[1.0, 0.0, 1.0, 3.0, 0.1]])
r3 = capsule_overlap(a3, b3)
assert r3[0, 0] == True, "Crossing spines → overlap"
print("Test 3 passed: perpendicular crossing capsules")

# ── Test 4: degenerate capsules (circles) ─────────────────────────────────
a4 = np.array([[0.0, 0.0, 0.0, 0.0, 2.0]])
b4 = np.array([[1.0, 0.0, 1.0, 0.0, 2.0]])
r4 = capsule_overlap(a4, b4)
assert r4[0, 0] == True, "Overlapping circles"
c4 = np.array([[10.0, 0.0, 10.0, 0.0, 1.0]])
r4b = capsule_overlap(a4, c4)
assert r4b[0, 0] == False, "Separated circles"
print("Test 4 passed: degenerate capsules (circles)")

# ── Test 5: large N=M=500 ─────────────────────────────────────────────────
rng = np.random.default_rng(42)
pts_a = rng.uniform(-50, 50, (500, 4))
pts_b = rng.uniform(-50, 50, (500, 4))
caps_a5 = np.hstack([pts_a, rng.uniform(0.5, 3.0, (500, 1))])
caps_b5 = np.hstack([pts_b, rng.uniform(0.5, 3.0, (500, 1))])
t0 = time.time()
r5 = capsule_overlap(caps_a5, caps_b5)
elapsed = time.time() - t0
assert r5.shape == (500, 500), f"Shape: {r5.shape}"
assert elapsed < 3.0, f"Too slow: {elapsed:.2f}s"
print(f"Test 5 passed: N=M=500 ({elapsed:.3f}s)")

print("\nAll tests passed!")